# Ferroload — Core Flows

A focused tour of the everyday `ferroload` API on an existing dataset:

1. Load an existing dataset
2. Map a **text** column (metadata → new annotation)
3. Map an **image** column (decoded image → new tensor modality)
4. Overwrite a column (re-map with `resume=False`)
5. Inspect the dataset
6. Subset the dataset
7. Fast one-call loader (`make_loader`)
8. Convert a `Dataset` object into a PyTorch `DataLoader`

> `map` binds inputs **positionally** and runs `fn` **once per sample** by default
> (`batched=False`), so map functions are generic and reusable across datasets.
> Outputs are written as **additive layers** in the dataset root — the base data
> is never rewritten, so running cells 2–4 permanently enriches it on disk.

## 0. Imports

In [ ]:
import numpy as np
import ferroload
from ferroload import make_loader, loader as fl

print("ferroload", ferroload.__version__)

## 1. Load an existing dataset

`Dataset.open(root)` opens the base data **plus any enrichment layers** already
registered in the manifest.

In [ ]:
ROOT = "/Users/midhun/pexelvideos/"
ds = ferroload.Dataset.open(ROOT)

print(ds)                                   # Dataset(len=..., name=...)
print("modalities:", list(ds.modalities().keys()))
print("meta keys :", sorted(ds[0]["meta"].keys()))

## 2. Map a text column

Derive a new **annotation** from existing string metadata. Annotation outputs
are stored inline in the layer index and merged into `meta` on read — no shards.
`fn` is bound to `inputs` **positionally** and runs **once per sample** (default
`batched=False`), so it stays generic — no column names live inside the function.

In [ ]:
def slug_from_title(title):                  # one sample's title -> its slug (reusable, generic)
    return title.split("·")[0].strip().lower().replace(" ", "-")[:40]

ds = ds.map(slug_from_title, inputs=["title"],
            outputs={"slug": ferroload.Annotation()},  # text -> meta
            name="slug", batch_size=16)

print("slug[0]:", ds.get(0)["meta"]["slug"])
print("slug[1]:", ds.get(1)["meta"]["slug"])

## 3. Map an image column

Image-codec modalities are **decoded to `[H, W, C]` uint8 arrays in Rust** before
`fn` sees them (in parallel, even in per-sample mode). A `Modality("npy")` output
becomes a new `.npy`-backed tensor modality joined on `sample_id`; read it back
with `read_array`. For a batched ML model, use `batched=True` and return a list.

In [ ]:
def mean_color(img):                         # one [H,W,C] uint8 image -> its [C] mean color
    return img.mean(axis=(0, 1)).astype("float32")

ds = ds.map(mean_color, inputs=["image"],
            outputs={"emb": ferroload.Modality("npy")},        # tensor -> new modality
            name="emb", batch_size=8)

print("modalities now:", list(ds.modalities().keys()))         # 'emb' joined in
print("emb[0]:", ds.read_array(0, "emb"))

## 4. Overwrite a column

`map` is idempotent by default (`resume=True` skips `sample_id`s already present).
To **recompute and overwrite** a layer-contributed column, pass `resume=False`:
every row is recomputed and the later value wins on read.

> Only **layer-added** columns can be overwritten this way — base modalities and
> base metadata always take precedence over layers.

In [ ]:
def slug_v2(title):
    return title.strip().upper()[:20]

ds = ds.map(slug_v2, inputs=["title"],
            outputs={"slug": ferroload.Annotation()},
            name="slug", resume=False)                 # recompute all -> overwrite

print("slug[0] after overwrite:", ds.get(0)["meta"]["slug"])

## 5. Inspect the dataset

Introspection comes straight from the binding — no manual `manifest.json` parsing.
`verify()` checks every referenced shard member reads back to its declared length.

In [ ]:
print("len        :", len(ds), "| shards:", ds.num_shards())
print("name/version:", ds.name, ds.version)
print("modalities :", list(ds.modalities().keys()))
print("schema cols:", [c["name"] for c in ds.schema()])

s = ds[0]                                   # == ds.get(0)
print("sample keys:", sorted(s.keys()))
print("meta keys  :", sorted(s["meta"].keys()))

p = ds.get(0, ["emb"])                       # projection -> only 'emb' fetched
print("projected  :", sorted(p.keys()))
print("verify     :", ds.verify(), "samples OK")

## 6. Subset the dataset

`subset(where_sql)` filters on **inline base metadata** (string `=`/`!=`, numeric
comparisons, `AND`/`OR`/`NOT`, and `<modality>_present` flags). It returns a new
(index-remapped) `Dataset` view by default, or the raw `list[int]` of ids with
`return_indices=True`.

In [ ]:
ids = ds.subset("family_friendly = 'yes'", return_indices=True)
print("family-friendly ids:", len(ids), "| first 5:", ids[:5])

sub = ds.subset("requires_subscription = 'no' AND video_present")  # -> Dataset view
print("subset view:", sub, "| len:", len(sub))
print("subset[0] meta title:", sub.get(0)["meta"]["title"][:50])

## 7. Fast one-call loader

`make_loader` bundles open + dataset + deterministic sampler + background
prefetch (parallel Rust decode with the GIL released). Pass a flat `columns=[...]`
list and each name's kind is **resolved from the manifest** — image/video codecs
decode, `.npy` columns load as arrays, other modalities pass as raw bytes, and
names that aren't modalities are treated as metadata. (Video decode needs a build
with `--features video`.) Call `set_epoch(e)` each epoch to reshuffle.

In [ ]:
dl = make_loader(ROOT, batch_size=8, columns=["image", "video", "title", "duration"],
                 resize=(64, 64), out="numpy")          # types resolved from the manifest
print("batches:", len(dl))

dl.set_epoch(0)
for batch in dl:
    print("image batch:", batch["image"].shape, "| titles   :", len(batch["title"]))
    print("video batch:", batch["video"].shape, "| durations:", len(batch["duration"]))
    break

## 8. Convert a `Dataset` object to a PyTorch `DataLoader`

`ds.torch(...)` returns a map-style `FerroTorchDataset` (yields per-sample dicts);
hand it to a standard `DataLoader` with a `FerroSampler`. The batched fast path
(`__getitems__`) is used automatically, so the default collate just works. The
same `columns=[...]` resolution works here too.

In [ ]:
from torch.utils.data import DataLoader

tds = ds.torch(columns=["image", "duration"], resize=(64, 64))  # FerroTorchDataset (out='torch')
dataloader = DataLoader(tds, batch_size=8, sampler=fl.FerroSampler(len(ds), seed=0))

batch = next(iter(dataloader))
print("torch batch keys:", list(batch.keys()))
print("image tensor    :", batch["image"].shape, batch["image"].dtype)